# Week 4 - Text Representation I : Classic Method (1/2)
**Notebook 1 : Bag of Words**

**Unstructured Data Analysis (2026-2)** · Professor: Misuk Kim · Teaching Assistant: Hojin Son

Week 3 turned raw text into clean tokens. The lecture *Text Representation I - Classic Method* asks the next question:

> **How to convert an unstructured text into a vector / matrix form** so that machine learning algorithms
> based on a vector space can be applied?

This notebook implements **chapter 1, Bag of Words**, of that lecture:

| Lecture slide | What we do in code |
|---|---|
| Bag of Words: Idea | build the term-document matrix of `S1`, `S2` by hand |
| Binary vs Frequency representation | `CountVectorizer(binary=True)` vs the default |
| "We cannot reconstruct the original text" | check that word order is really gone |
| Text Preprocessing | see how lowercasing / punctuation / numbers change the vocabulary |
| Stop Words | NLTK's English list, and a custom list for Korean |
| *Extension* | the same matrix on 2,000 real documents, and the **sparsity** it produces |
| *Extension* | Korean text: why a morphological analyzer is needed |

> **Note on orientation.** The lecture draws a **term-document matrix** (terms as rows, documents as columns).
> scikit-learn returns the transpose, a **document-term matrix** (documents as rows, terms as columns).
> They hold the same information; only the axes are swapped.

> **How to use this notebook in Colab**: `File ▸ Save a copy in Drive`, then run the cells from top to bottom (`Shift + Enter`).

## &nbsp;0. Setup
We use the NLTK **movie_reviews** corpus (2,000 English movie reviews) and, for the Korean part,
the **Daum movie review** dataset loaded from the course repository.

> ⚠️ Run this cell **once** per session. As in Week 3, the tokenizer resources have names ending with `_tab` since NLTK 3.9.

In [ ]:
import nltk
import numpy as np
import pandas as pd

for r in ['movie_reviews', 'punkt', 'punkt_tab', 'stopwords']:
    nltk.download(r, quiet=True)

print("NLTK version:", nltk.__version__)
print("All resources downloaded.")

## &nbsp;1. Bag of Words by Hand
The lecture example:

> **S1**: John likes to watch movies. Mary likes too.
> **S2**: John also likes to watch football game.

A **bag of words** keeps only *which* words occur and *how often* - the words are treated as an
**unordered collection**, each word an independent symbol in a discrete space.

Let's build the **term-document matrix** of these two sentences exactly as on the lecture slide.

In [ ]:
from nltk.tokenize import RegexpTokenizer

S1 = "John likes to watch movies. Mary likes too."
S2 = "John also likes to watch football game."

tok = RegexpTokenizer(r"[\w']+")

docs = [tok.tokenize(S1.lower()), tok.tokenize(S2.lower())]
print('S1 tokens:', docs[0])
print('S2 tokens:', docs[1])

In [ ]:
# The vocabulary is the set of all words, in a FIXED order.
# The lecture lists them in order of first appearance, so we do the same.

vocab = []
for d in docs:
    for w in d:
        if w not in vocab:
            vocab.append(w)

print('vocabulary:', vocab)
print('|V| =', len(vocab))

In [ ]:
def frequency_vector(doc, vocab):
    return [doc.count(w) for w in vocab]          # how many times each vocabulary word occurs


def binary_vector(doc, vocab):
    return [1 if w in doc else 0 for w in vocab]  # does it occur at all?


tdm = pd.DataFrame({
    'Word': vocab,
    'S1 (binary)': binary_vector(docs[0], vocab),
    'S2 (binary)': binary_vector(docs[1], vocab),
    'S1 (frequency)': frequency_vector(docs[0], vocab),
    'S2 (frequency)': frequency_vector(docs[1], vocab),
})
tdm

**🤔 Quick check**
Find the single cell where the two representations differ.
`likes` occurs **twice** in S1, so the frequency representation writes `2` where the binary one writes `1`.
That one number is what introduces **term frequency**, which becomes one of the main ingredients
of the weighting schemes in Notebook 2.

### &nbsp;1-1. The order is gone
The lecture makes the point twice:

* *John is quicker than Mary* = *Mary is quicker than John* in a BoW representation
* From the vector `(…, John 1, …, loves 1, …, Mary 1, …)` we cannot tell whether the original text was
  *John loves Mary* or *Mary loves John*.

Let's verify it.

In [ ]:
a = tok.tokenize("John is quicker than Mary".lower())
b = tok.tokenize("Mary is quicker than John".lower())

v = sorted(set(a) | set(b))
print('vocabulary   :', v)
print('vector of a  :', frequency_vector(a, v))
print('vector of b  :', frequency_vector(b, v))
print('identical?   :', frequency_vector(a, v) == frequency_vector(b, v))

**So where does the lost order go?** Nowhere, for now - Bag of Words simply does not keep it.
We buy a little of it back in **Notebook 2**, with **N-grams**.

Before that, a more basic question. A term-document matrix is fixed by two decisions:
**which words get a column**, and **what goes in each cell**. Section 1 played with the cells -
binary or frequency. Sections 2 and 3 decide the columns, and those decisions are made
*before* anything is counted.

From here on we leave `S1` / `S2` behind and measure the effect on a real corpus:
2,000 movie reviews from NLTK.

## &nbsp;2. Preprocessing Changes the Vocabulary
Before counting anything we must decide what a "word" is. The lecture lists three decisions:

| decision | lecture point |
|---|---|
| lowercase | `They` vs `they` are different words in many systems |
| punctuation | punctuation carries little information → usually removed |
| numbers | not critical in some domains, critical in others → decide per domain |

Every one of those decisions changes the **columns** of the matrix. Let's measure the effect on a real corpus.

In [ ]:
from nltk.corpus import movie_reviews

reviews = [movie_reviews.raw(fileid) for fileid in movie_reviews.fileids()]

print('#documents:', len(reviews))
print('#categories:', movie_reviews.categories())
print()
print(reviews[0][:300])

In [ ]:
from nltk.tokenize import word_tokenize

sample = ("The Matrix (1999) is a great movie. THE MATRIX changed sci-fi forever! "
          "It cost $63,000,000 and earned 463 million.")

variants = {
    'raw word_tokenize          ': word_tokenize(sample),
    'lowercase                  ': word_tokenize(sample.lower()),
    "no punctuation  [\\w']+     ": RegexpTokenizer(r"[\w']+").tokenize(sample.lower()),
    "no numbers      [a-z']+    ": RegexpTokenizer(r"[a-z']+").tokenize(sample.lower()),
    "3+ letters      [a-z']{3,} ": RegexpTokenizer(r"[a-z']{3,}").tokenize(sample.lower()),
}

for name, tokens in variants.items():
    print(f'{name} tokens={len(tokens):3d}  distinct={len(set(tokens)):3d}')

print()
print('raw       :', variants['raw word_tokenize          '][:12])
print('lowercase :', variants['lowercase                  '][:12])
print('no numbers:', variants["no numbers      [a-z']+    "][:12])

Two numbers, two different things:

* **`tokens`** - how many words we counted. This is what goes **into** the cells.
* **`distinct`** - how many different words there are. This is the number of **columns** of the matrix.

Lowercasing keeps `tokens` at 25 - nothing is deleted, `matrix` still occurs twice in the sample.
What it changes is `distinct`, 24 -> 22: the columns `Matrix` and `MATRIX` become one column `matrix`,
and so do `The` and `THE`. That is the point of this section: preprocessing rewrites the **columns**,
not the counts.

Then `(`, `.`, `!`, `$` disappear, and the numbers go last. Note what removing punctuation costs:
`63,000,000` is not deleted but **shattered** into `63`, `000`, `000` - one number becomes three columns.

In [ ]:
# The same decisions on the whole corpus, measured as vocabulary size

for name, pattern in [("[\\w']+     (words + digits)", r"[\w']+"),
                      ("[a-z']+     (no digits)   ", r"[a-z']+"),
                      ("[a-z']{3,}  (3+ letters)  ", r"[a-z']{3,}")]:
    tk = RegexpTokenizer(pattern)
    vocab = set()
    for doc in reviews:
        vocab.update(tk.tokenize(doc.lower()))
    print(f'{name} -> {len(vocab):,} distinct words')

# Note: this corpus is already lowercased, so the lowercase step changes nothing here.
# On raw text, lowercasing merges case variants such as "Movie" and "movie",
# which can reduce the vocabulary considerably.

**🤔 Quick check**
The distinct count is the number of **columns** your document-term matrix would have.
Preprocessing is not a cosmetic step - it *is* the representation.

## &nbsp;3. Stop Words
**Stop words** are very frequent, mainly functional words that usually carry
**little information for telling documents apart** in a given task.
The lecture gives three reference lists:

| list | size |
|---|---|
| SMART (System for the Mechanical Analysis and Retrieval of Text) | 571 |
| MySQL full-text | 543 |
| Korean (ranks.nl) | 677 |

NLTK ships its own English list. Korean has no single standard list, so a **custom** list is the usual approach.

In [ ]:
from nltk.corpus import stopwords

english_stops = set(stopwords.words('english'))

print('NLTK English stop words:', len(english_stops))
print(sorted(english_stops)[:20])

In [ ]:
tokenizer = RegexpTokenizer(r"[\w']{3,}")        # one of the patterns we compared in Week 3: 3+ word characters

tokens_all = tokenizer.tokenize(reviews[0].lower())
tokens_kept = [t for t in tokens_all if t not in english_stops]

print('before stop-word removal:', len(tokens_all), 'tokens,', len(set(tokens_all)), 'distinct')
print('after  stop-word removal:', len(tokens_kept), 'tokens,', len(set(tokens_kept)), 'distinct')
print()
print('removed:', sorted(set(tokens_all) - set(tokens_kept))[:20])

**🤔 Why remove them before building the matrix, and not after?**
A stop word would otherwise take a column of its own and, because it occurs in nearly every document,
it would occupy a high-count dimension while contributing little to distinguishing documents.
Notebook 2 shows the *other* way to deal with this: leave the word in, but give it a small **weight**.

## &nbsp;4. The Term-Document Matrix of a Real Corpus
Now the same procedure as section 1, but on all 2,000 reviews. Two steps:

1. count every word of the corpus and keep the **top 1,000** as the feature list (a fixed order)
2. for every document, read the counts in that order

In [ ]:
documents = [[t for t in tokenizer.tokenize(doc.lower()) if t not in english_stops]
             for doc in reviews]

word_count = {}
for doc in documents:
    for word in doc:
        word_count[word] = word_count.get(word, 0) + 1

sorted_features = sorted(word_count, key=word_count.get, reverse=True)

print('#distinct words in the corpus:', len(sorted_features))
for w in sorted_features[:10]:
    print(f"   {w:<10} {word_count[w]}")

In [ ]:
word_features = sorted_features[:1000]     # the 1,000 columns of our matrix

print(word_features[:20])

In [ ]:
def document_features(document, word_features):

    counts = {}
    for word in document:                             # 1) count the words of this document
        counts[word] = counts.get(word, 0) + 1

    return [counts.get(w, 0) for w in word_features]  # 2) read them in the fixed feature order


# the tiny test from section 1, reused
print(document_features(['two', 'two', 'couples'], ['one', 'two', 'teen', 'couples', 'solo']))

In [ ]:
feature_sets = [document_features(d, word_features) for d in documents]

print('#documents:', len(feature_sets), ' #features:', len(feature_sets[0]))
print()
for i in range(15):
    print(f'({word_features[i]}, {feature_sets[0][i]})', end=', ')

### &nbsp;4-1. Sparsity
The lecture's own warning about the vector space: *"Very high dimensional"* and *"Sparseness: most entries are zero"*.
Let's put numbers on it.

In [ ]:
print('first 20 values:', feature_sets[0][:20])
print('last  20 values:', feature_sets[0][-20:])

zeros = feature_sets[0].count(0)
print(f'\n{zeros} of {len(feature_sets[0])} values are 0  ->  {zeros / len(feature_sets[0]):.1%} of the vector is empty')

In [ ]:
# And that is with only 1,000 features. With the full vocabulary:

full = len(sorted_features)
used = len(set(documents[0]))
print(f'full vocabulary        : {full:,} columns')
print(f'distinct words in doc 0: {used:,}')
print(f'-> {used / full:.2%} of the row would be non-zero')

## &nbsp;5. The Same Thing with Scikit-learn
`CountVectorizer` does tokenization, vocabulary building and vectorization in one object.

| parameter | meaning |
|---|---|
| `vocabulary` | use a feature list you built yourself |
| `max_features` | keep only the *n* most frequent words |
| `min_df`, `max_df` | ignore words that appear in too few / too many documents |
| `binary` | `True` gives the **binary representation** of section 1 |
| `tokenizer` | your own tokenizer function (required for Korean) |
| `ngram_range` | also use word pairs / triples (Notebook 2) |

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(vocabulary=word_features)
reviews_cv = cv.fit_transform(reviews)

print(cv.get_feature_names_out()[:15])      # the features used by the vectorizer
print(word_features[:15])                   # our own list - same words, same order
print()
print('manual  :', feature_sets[0][:15])
print('sklearn :', reviews_cv.toarray()[0, :15])

# Almost the same - but not exactly. Look for the positions that differ.

### &nbsp;5-1. Same vocabulary, different tokenizer
`CountVectorizer` uses **its own** default tokenizer, the pattern `\b\w\w+\b`: it lowercases the text,
treats every non-word character as a separator, and keeps only runs of **two or more** word characters.
So `film's` is effectively reduced to `film` - the apostrophe separates, and the one-character token `s`
is dropped by the pattern - while our `RegexpTokenizer(r"[\w']{3,}")` keeps `film's` as a single token.

The fix is the same one we will need for Korean: pass our own tokenizer function.

In [ ]:
diff = [(word_features[i], feature_sets[0][i], reviews_cv.toarray()[0, i])
        for i in range(15) if feature_sets[0][i] != reviews_cv.toarray()[0, i]]
print('feature, manual, sklearn ->', diff)

In [ ]:
def en_tokenizer(doc):
    # exactly the tokenization we used in section 4
    return [t for t in tokenizer.tokenize(doc.lower()) if t not in english_stops]


cv = CountVectorizer(vocabulary=word_features, tokenizer=en_tokenizer, token_pattern=None)
reviews_cv = cv.fit_transform(reviews)

print('manual  :', feature_sets[0][:15])
print('sklearn :', reviews_cv.toarray()[0, :15])
print('identical:', (reviews_cv.toarray()[0] == np.array(feature_sets[0])).all())

### &nbsp;5-2. Binary vs frequency, on the real corpus
`binary=True` reproduces the left-hand table of the lecture slide.

In [ ]:
cv_bin = CountVectorizer(vocabulary=word_features, tokenizer=en_tokenizer,
                         token_pattern=None, binary=True)
reviews_bin = cv_bin.fit_transform(reviews)

print('frequency:', reviews_cv.toarray()[0, :15])
print('binary   :', reviews_bin.toarray()[0, :15])
print()
print('total counts  :', reviews_cv.sum())
print('non-zero cells:', reviews_bin.sum(), '(= number of (document, word) pairs)')

### &nbsp;5-3. Why a *sparse* matrix?
`CountVectorizer` returns a `scipy` **sparse matrix**: only the non-zero values are stored.
With a full NumPy array a realistic corpus would not fit into memory.

In [ ]:
# ask the matrix for its own item size instead of hard-coding one
dense_mb = (reviews_cv.shape[0] * reviews_cv.shape[1]
            * reviews_cv.dtype.itemsize / 1024**2)

print('#dtype              :', reviews_cv.dtype, f'({reviews_cv.dtype.itemsize} bytes per value)')
print('#type of the matrix :', type(reviews_cv))
print('#shape              :', reviews_cv.shape, ' (documents x features)')
print(f'stored values (non-zero): {reviews_cv.nnz:,}')
print(f'all values              : {reviews_cv.shape[0] * reviews_cv.shape[1]:,}')
print(f'density                 : {reviews_cv.nnz / (reviews_cv.shape[0] * reviews_cv.shape[1]):.2%}')
print(f'dense size would be     : {dense_mb:.1f} MB')

## &nbsp;6. Korean Text
The lecture notes that stop words are **natural-language dependent** (`…습니다`, `…로서(써)`, `…를`).
Korean needs one more step before that: words carry **particles** (조사) and **endings** (어미),
so `영화가`, `영화는`, `영화를` would each take a column of their own.

We run a morphological analyzer - `Okt` from **KoNLPy** - and pass it to `CountVectorizer` as a custom tokenizer.

Dataset: **Daum movie reviews** (14,725 reviews with a 1-10 rating), loaded directly from the course repository.

In [ ]:
url = "https://raw.githubusercontent.com/snhzyn/2026-2-Unstructured-Data-Analysis/main/week04-text-representation-1/data/daum_movie_review.csv"
df = pd.read_csv(url)

print(df.shape)
df.head(10)

In [ ]:
# Without a morphological analyzer: CountVectorizer's default pattern treats punctuation
# as a separator and keeps runs of 2+ word characters - it knows nothing about 조사/어미

daum_cv = CountVectorizer(max_features=1000)
daum_DTM = daum_cv.fit_transform(df.review)

print(daum_cv.get_feature_names_out()[:50])
# Look at the features: '영화가', '영화는', '영화를' are counted as three different words.

In [ ]:
# KoNLPy needs a Java runtime. In Colab the line below is usually enough;
# if you get a JVM error, run:  !apt-get install -y openjdk-17-jdk-headless -qq

!pip install -q konlpy

In [ ]:
from konlpy.tag import Okt

okt = Okt()

print('# morphemes :', okt.morphs(df.review[1]))
print('# nouns     :', okt.nouns(df.review[1]))
print('# POS tags  :', okt.pos(df.review[1]))

In [ ]:
KOREAN_STOPS = {'정말', '너무', '진짜', '그냥'}   # a custom list - Korean has no standard one


def my_tokenizer(doc):
    # keep only content words, then drop our own stop words
    tokens = [t for t, pos in okt.pos(doc) if pos in ['Noun', 'Verb', 'Adjective']]
    return [t for t in tokens if t not in KOREAN_STOPS]


print("my_tokenizer:", my_tokenizer(df.review[1]))

In [ ]:
# This cell takes about a minute: the analyzer runs over all 14,725 reviews.

daum_cv = CountVectorizer(max_features=1000, tokenizer=my_tokenizer, token_pattern=None)
daum_DTM = daum_cv.fit_transform(df.review)

print(daum_cv.get_feature_names_out()[:50])
print()
print(repr(daum_DTM))

In [ ]:
# the count vector of the second review, showing only the words that actually occur
for word, count in zip(daum_cv.get_feature_names_out(), daum_DTM[1].toarray()[0]):
    if count > 0:
        print(word, ':', count, end=', ')

**🤔 Quick check**
Compare the two feature lists. With `Okt` the variants `영화가 / 영화는 / 영화를` collapse into the single feature `영화`,
so the same 1,000 columns now cover far more of the corpus. Choosing the tokenizer **is** part of the representation -
exactly the point of section 2, in a language where you cannot avoid it.

### &nbsp;6-1. Morphological analysis is not stop-word removal
The two steps are independent. `Okt` merged the surface forms into `영화`; whether `영화` is *useful*
is a separate, **task-dependent** decision. In a corpus of movie reviews it appears almost everywhere,
so it behaves like a stop word for this domain - exactly the `idf` argument of Notebook 2.

In [ ]:
DOMAIN_STOPS = KOREAN_STOPS | {'영화'}          # drop it only if the task calls for it


def domain_tokenizer(doc):
    tokens = [t for t, pos in okt.pos(doc) if pos in ['Noun', 'Verb', 'Adjective']]
    return [t for t in tokens if t not in DOMAIN_STOPS]


# pick a review that actually mentions 영화
sample = next(r for r in df.review if '영화' in r and len(r) < 30)

print('review   :', sample)
print('keep 영화 :', my_tokenizer(sample))
print('drop 영화 :', domain_tokenizer(sample))
print()
print("'영화' is among the 1,000 features :", '영화' in daum_cv.get_feature_names_out())

### ✅ What we practiced
* Term-document matrix by hand : fixed vocabulary order, **binary** vs **frequency** representation
* Word order is genuinely lost - the original text cannot be reconstructed
* Preprocessing decisions (lowercase, punctuation, numbers, token length) change the columns of the matrix
* Stop words : NLTK's English list, a custom list for Korean
* The same matrix on 2,000 documents : `CountVectorizer`, `binary=True`, sparse storage
* **Sparsity** : 85 % of a 1,000-dimensional row is 0, and it gets worse with the full vocabulary
* Korean : `Okt` morphological analysis as a custom `tokenizer`

Every cell of this matrix is still a raw count, so `film` - which appears in almost every review - looks
as important as a word that appears in only one.
Next notebook: **Word Weighting & N-grams**, which fixes exactly that.